In [5]:
import tensorflow as tf
from tensorflow.keras.models import load_model
import pickle
import pandas as pd
import numpy as np


I0000 00:00:1773482708.226964  353331 cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.
I0000 00:00:1773482708.387738  353331 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
I0000 00:00:1773482715.693066  353331 cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.


In [6]:
# Load the trained model, scaler pickel, onehot
model=load_model('model.h5')

## load the encoder & scaler
with open('label_encoder_gender.pkl','rb') as file:
    label_encoder_gender=pickle.load(file)


with open('ohe_geo.pkl','rb') as file:
    ohe_encoder_geo=pickle.load(file)

with open('scaler.pkl','rb') as file:
    scaler=pickle.load(file)

E0000 00:00:1773482753.145126  353331 cuda_platform.cc:52] failed call to cuInit: INTERNAL: CUDA error: Failed call to cuInit: UNKNOWN ERROR (303)


In [25]:
input_data={
    'CreditScore': 600,
    'Geography':'France',
     'Gender':'Male',
     'Age':40,
     'Tenure':3,
     'Balance':60000,
     'NumOfProducts':2,
     'HasCrCard':1,
     'IsActiveMember':1,
     'EstimatedSalary':50000
}

In [30]:
geo_encoded= ohe_encoder_geo.transform([[input_data['Geography']]]).toarray()
geo_encoded_df=pd.DataFrame(geo_encoded, columns=ohe_encoder_geo.get_feature_names_out(['Geography']))

geo_encoded_df

/home/vwagh/git/udemy/project/churn_modelling/venv/lib/python3.11/site-packages/sklearn/utils/validation.py:2691: UserWarning: X does not have valid feature names, but OneHotEncoder was fitted with feature names
  warnings.warn(


,Geography_France,Geography_Germany,Geography_Spain
0,1.0,0.0,0.0


In [31]:
# Solved: 'dict' object has no attribute 'reset_index'
input_df=pd.DataFrame([input_data]) # If you run 2nd time it throw an error.
input_df

,CreditScore,Geography,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary
0,600,France,Male,40,3,60000,2,1,1,50000


In [32]:
## Encode categorical variables
input_df['Gender']=label_encoder_gender.transform(input_df['Gender'])
input_df

,CreditScore,Geography,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary
0,600,France,1,40,3,60000,2,1,1,50000


In [35]:
# combine one hot encoding column with input data

input_df=pd.concat([input_df.drop('Geography', axis=1), geo_encoded_df],axis=1)
input_df

,CreditScore,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Geography_France,Geography_Germany,Geography_Spain,Geography_France,Geography_Germany,Geography_Spain
0,600,1,40,3,60000,2,1,1,50000,1.0,0.0,0.0,1.0,0.0,0.0


In [37]:
## Scaling the input data
input_scaled=scaler.transform(input_df)
input_scaled

ValueError: The feature names should match those that were passed during fit.
Feature names must be in the same order as they were in fit.


In [38]:
scaler

,"copy copy: bool, default=TrueIf False, try to avoid a copy and do inplace scaling instead.This is not guaranteed to always work inplace; e.g. if the data isnot a NumPy array or scipy.sparse CSR matrix, a copy may still bereturned.",True
,"with_mean with_mean: bool, default=TrueIf True, center the data before scaling.This does not work (and will raise an exception) when attempted onsparse matrices, because centering them entails building a densematrix which in common use cases is likely to be too large to fit inmemory.",True
,"with_std with_std: bool, default=TrueIf True, scale the data to unit variance (or equivalently,unit standard deviation).",True
